In [ ]:
# --- 0. Install Libraries (Google Colab Environment) ---
# If you are running this code for the first time in Colab,
# uncomment and run the lines below to install the necessary libraries.
# !pip install meteostat
# !pip install torch numpy pandas scikit-learn matplotlib
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from datetime import datetime, date, timedelta
import os

# --- 1. Data Preparation (Using Meteostat) ---

# Set data period (from past to present)
START_DATE = "2021-01-01"
END_DATE = date.today().strftime("%Y-%m-%d")
print(f"Data collection period: {START_DATE} ~ {END_DATE}")

def fetch_from_meteostat(start_date, end_date):
    """Fetches data from the Meteostat library."""
    print("Fetching data from Meteostat library...")
    try:
        from meteostat import Point, Daily
        start = datetime.strptime(start_date, "%Y-%m-%d")
        end = datetime.strptime(end_date, "%Y-%m-%d")
        vientiane = Point(17.97, 102.61, 70)
        data = Daily(vientiane, start, end).fetch()
        if data.empty:
            raise ValueError("Failed to fetch data from Meteostat.")
        df = data[['tavg']].copy()
        df.rename(columns={'tavg': 'temperature_2m_mean'}, inplace=True)
        print(" -> Data successfully received from Meteostat!")
        return df
    except Exception as e:
        print(f" -> Meteostat failed: {e}")
        return None

# Fetch data using Meteostat
temperature_data = fetch_from_meteostat(START_DATE, END_DATE)

# Exit if data collection fails
if temperature_data is None or temperature_data.empty:
    print("\nFailed to fetch data from all providers. Exiting the program.")
    exit()

temperature_data.dropna(inplace=True)
print(f"Using a total of {len(temperature_data)} days of data for training.")
print("-" * 30)


# --- 2. Data Preprocessing ---
SEQ_LENGTH = 60  # Number of past days to use for prediction (increased to 60)

# Scale the entire dataset
scaler = MinMaxScaler()
scaled_data = scaler.fit_transform(temperature_data[['temperature_2m_mean']])

def create_sequences(data, seq_length):
    xs, ys = [], []
    for i in range(len(data) - seq_length):
        x = data[i:(i + seq_length)]
        y = data[i + seq_length]
        xs.append(x)
        ys.append(y)
    return np.array(xs), np.array(ys)

# Create training sequences using the entire dataset
X_train, y_train = create_sequences(scaled_data, SEQ_LENGTH)

# Convert to PyTorch tensors
X_train = torch.tensor(X_train, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.float32)

print(f"Training sequences X: {X_train.shape}, y: {y_train.shape}")
print("-" * 30)


# --- 3. Define LSTM Model ---
class WeatherLSTM(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, output_size):
        super(WeatherLSTM, self).__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        out, _ = self.lstm(x)
        out = self.fc(out[:, -1, :])
        return out

INPUT_SIZE = 1
HIDDEN_SIZE = 64
NUM_LAYERS = 2
OUTPUT_SIZE = 1
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = WeatherLSTM(INPUT_SIZE, HIDDEN_SIZE, NUM_LAYERS, OUTPUT_SIZE).to(device)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)


# --- 4. Model Training ---
print("\n--- Starting Model Training ---")
epochs = 150 # Increase epochs for more training
batch_size = 32
train_dataset = torch.utils.data.TensorDataset(X_train, y_train)
train_loader = torch.utils.data.DataLoader(dataset=train_dataset, batch_size=batch_size, shuffle=True)

for epoch in range(epochs):
    model.train()
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    if (epoch + 1) % 10 == 0:
        print(f'Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.6f}')
print("--- Model Training Complete ---")
print("-" * 30)

# --- 5. Save Trained Model and Preprocessing Tools ---
print("--- Saving model and scaler started ---")
torch.save(model.state_dict(), 'weather_forecast_model.pth')
# Save the MinMaxScaler object to be reused for prediction
import joblib
joblib.dump(scaler, 'scaler.pkl')

# Save data and index
temperature_data.to_csv('temperature_data.csv')
print("--- Saving model and scaler complete ---")


In [ ]:
# --- 0. Install Libraries (Google Colab Environment) ---
# If you are running this code for the first time in Colab,
# uncomment and run the lines below to install the necessary libraries.
# !pip install torch numpy pandas scikit-learn matplotlib ipywidgets
# Import necessary libraries
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime, date, timedelta
import joblib
from sklearn.preprocessing import MinMaxScaler
from IPython.display import display, clear_output
import ipywidgets as widgets

# --- 1. Define Model Class (Required to load the saved model) ---
class WeatherLSTM(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, output_size):
        super(WeatherLSTM, self).__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        out, _ = self.lstm(x)
        out = self.fc(out[:, -1, :])
        return out

# --- 2. Load Trained Model and Preprocessing Tools ---
print("--- Loading model and scaler started ---")
INPUT_SIZE = 1
HIDDEN_SIZE = 64
NUM_LAYERS = 2
OUTPUT_SIZE = 1
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = WeatherLSTM(INPUT_SIZE, HIDDEN_SIZE, NUM_LAYERS, OUTPUT_SIZE).to(device)
model.load_state_dict(torch.load('weather_forecast_model.pth', map_location=device))
model.eval()

# Load the saved scaler
scaler = joblib.load('scaler.pkl')

# Load the saved data
temperature_data = pd.read_csv('temperature_data.csv', index_col=0, parse_dates=True)
scaled_data = scaler.fit_transform(temperature_data[['temperature_2m_mean']])

print("--- Loading model and scaler complete ---")
print("-" * 30)


# --- 3. Future Weather Prediction and Visualization (Interacting with Slider) ---

def update_forecast(days_to_predict):
    """
    Function to perform prediction and plot graphs based on slider value.
    Args:
        days_to_predict (int): Number of days to predict
    """
    clear_output(wait=True)
    print(f"\n--- Starting {days_to_predict}-day weather forecast ---")
    SEQ_LENGTH = 60

    # Last sequence data to be used as the starting point for prediction
    last_sequence = scaled_data[-SEQ_LENGTH:]
    current_sequence = torch.tensor(last_sequence, dtype=torch.float32).reshape(1, SEQ_LENGTH, 1).to(device)

    future_predictions_scaled = []
    
    for _ in range(days_to_predict):
        with torch.no_grad():
            # Predict the next day
            next_pred_scaled = model(current_sequence)
            future_predictions_scaled.append(next_pred_scaled.cpu().numpy().flatten()[0])

            # Update the sequence: remove the oldest value and add the new prediction
            new_sequence_entry = next_pred_scaled.reshape(1, 1, 1)
            current_sequence = torch.cat((current_sequence[:, 1:, :], new_sequence_entry), dim=1)

    # Revert predictions to the original scale
    future_predictions = scaler.inverse_transform(np.array(future_predictions_scaled).reshape(-1, 1))

    # Create date index for the predicted period
    last_date = temperature_data.index[-1]
    future_dates = pd.date_range(start=last_date + timedelta(days=1), periods=days_to_predict)

    # Convert prediction results to a DataFrame
    future_df = pd.DataFrame(data=future_predictions, index=future_dates, columns=['temperature_2m_mean'])


    # ★★ Modified Visualization Section (Displaying actual and predicted data separately) ★★
    plt.figure(figsize=(15, 7))
    plt.title(f"Vientiane Temperature Forecast (Last 3 Years + {days_to_predict}-day Forecast)")
    plt.ylabel("Temperature (°C)")
    plt.xlabel("Date")
    plt.grid(True)

    # Plot historical actual data for the last 3 years
    three_years_ago = date.today() - timedelta(days=3*365)
    history_from_3_years_ago = temperature_data[temperature_data.index.date >= three_years_ago]

    if not history_from_3_years_ago.empty:
        plt.plot(history_from_3_years_ago.index, history_from_3_years_ago['temperature_2m_mean'], label='Historical Actual', color='royalblue')

    # Plot predicted data
    if not future_df.empty:
        plt.plot(future_df.index, future_df['temperature_2m_mean'], label='Future Forecast', color='darkorange', linestyle='--')

    # Check if there is data to display and add a legend
    if not history_from_3_years_ago.empty or not future_df.empty:
        plt.legend()
    else:
        plt.text(0.5, 0.5, 'No data to display', horizontalalignment='center', verticalalignment='center', transform=plt.gca().transAxes)

    plt.show()
    print("Please check the graph.")

# Create the slider widget
forecast_slider = widgets.IntSlider(
    value=60,
    min=30,
    max=120,
    step=1,
    description='Forecast Period (days):',
    disabled=False,
    continuous_update=False,
    orientation='horizontal',
    readout=True,
    readout_format='d'
)

# Link the slider to the function
widgets.interactive(update_forecast, days_to_predict=forecast_slider)
